In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p /content/drive/MyDrive/hm_project
%cd /content/drive/MyDrive/hm_project

/content/drive/MyDrive/hm_project


In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (5).json


{'kaggle (5).json': b'{"username":"afnakurikkal22","key":"63bd3eb4964ab63e60ba1567e965f17e"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f articles.csv

100% 4.26M/4.26M [00:00<00:00, 63.9MB/s]



In [6]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f customers.csv

100% 97.9M/97.9M [00:00<00:00, 139MB/s]



In [7]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f transactions_train.csv

100% 584M/584M [00:08<00:00, 69.3MB/s]



In [8]:
!unzip -o -q "*.zip"
!rm *.zip


3 archives were successfully processed.


In [9]:
!ls -lh

total 3.5G
-rw------- 1 root root  35M Jan 17  2022  articles.csv
drwx------ 2 root root 4.0K Nov  5 07:24  customer_feats.parquet
-rw------- 1 root root 198M Jan 17  2022  customers.csv
drwx------ 2 root root 4.0K Nov  5 07:26  item_popularity.parquet
-rw------- 1 root root   70 Nov  4 17:21 'kaggle (1).json'
-rw------- 1 root root   70 Nov  5 05:52 'kaggle (2).json'
-rw------- 1 root root   70 Nov  5 07:00 'kaggle (3).json'
-rw------- 1 root root   70 Nov  5 10:00 'kaggle (4).json'
-rw------- 1 root root   70 Nov  5 10:16 'kaggle (5).json'
-rw------- 1 root root   70 Nov  4 11:45  kaggle.json
-rw------- 1 root root 3.3G Jan 17  2022  transactions_train.csv


In [10]:
import pandas as pd
articles = pd.read_csv('articles.csv')
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [11]:
!pip install -q pyspark

In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("hm_project") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [13]:
transactions = spark.read.csv(
    "transactions_train.csv",
    header=True,
    inferSchema=True
)

In [14]:
!ls -lh /content

total 8.0K
drwx------ 5 root root 4.0K Nov  5 10:15 drive
drwxr-xr-x 1 root root 4.0K Oct 21 13:28 sample_data


In [15]:
transactions.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [16]:
transactions.show(5, truncate=False)

+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|t_dat     |customer_id                                                     |article_id|price               |sales_channel_id|
+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|663713001 |0.050830508474576264|2               |
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|541518023 |0.03049152542372881 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|505221004 |0.01523728813559322 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|685687003 |0.016932203389830508|2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|685687004 |0.016932203389830508|2 

In [17]:
print(transactions.count())

31788324


In [18]:
from pyspark.sql import functions as F

transactions.select(F.min("t_dat"), F.max("t_dat")).show()

+----------+----------+
|min(t_dat)|max(t_dat)|
+----------+----------+
|2018-09-20|2020-09-22|
+----------+----------+



In [19]:
transactions.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transactions.columns]
).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



In [20]:
print("Customers:", transactions.select("customer_id").distinct().count())
print("Articles:", transactions.select("article_id").distinct().count())

Customers: 1362281
Articles: 104547


In [21]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
articles = spark.read.csv("articles.csv", header=True, inferSchema=True)

In [22]:
customers.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers.columns]
).show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+



In [23]:
customers.select("age").describe().show()

+-------+-----------------+
|summary|              age|
+-------+-----------------+
|  count|          1356119|
|   mean|  36.386964565794|
| stddev|14.31362798162793|
|    min|               16|
|    max|               99|
+-------+-----------------+



In [24]:
print("Customers in customers.csv:", customers.count())
print("Articles in articles.csv:", articles.count())

Customers in customers.csv: 1371980
Articles in articles.csv: 105542


In [25]:
no_purchase = customers.join(
    transactions.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
)
print("Customers with no purchases:", no_purchase.count())

Customers with no purchases: 9699


In [26]:
unsold = articles.join(
    transactions.select("article_id").distinct(),
    on="article_id",
    how="left_anti"
)
print("Articles never bought:", unsold.count())

Articles never bought: 995


In [27]:
median_age = int(customers.approxQuantile("age", [0.5], 0.01)[0])
print("Median age:", median_age)

Median age: 32


In [28]:
customers_clean = (
    customers
    .withColumn("age_missing", F.col("age").isNull().cast("int"))
    .withColumn("fashion_news_frequency", F.upper(F.col("fashion_news_frequency")))
    .fillna({
        "age": median_age,
        "FN": 0,
        "Active": 0,
        "club_member_status": "UNKNOWN",
        "fashion_news_frequency": "NONE"
    })
)

In [29]:
customers_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers_clean.columns]
).show()

+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|customer_id| FN|Active|club_member_status|fashion_news_frequency|age|postal_code|age_missing|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|          0|  0|     0|                 0|                     0|  0|          0|          0|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+



In [30]:
customers_clean.groupBy("fashion_news_frequency").count().show()

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|               MONTHLY|   842|
|                  NONE|893722|
|             REGULARLY|477416|
+----------------------+------+



In [31]:
cutoff = "2020-09-16"
train = transactions.filter(F.col("t_dat") < cutoff)
test = transactions.filter(F.col("t_dat") >= cutoff)
print("Train rows:", train.count())
print("Test rows:", test.count())

Train rows: 31548013
Test rows: 240311


In [32]:
train = (
    train
    .withColumn("month", F.month("t_dat"))
    .withColumn("day_of_week", F.dayofweek("t_dat"))
    .withColumn("days_ago", F.datediff(F.lit(cutoff).cast("date"), F.col("t_dat")))
)

In [33]:
customer_feats = (
    train.groupBy("customer_id")
    .agg(
        F.count("*").alias("n_purchases"),
        F.countDistinct("article_id").alias("n_unique_items"),
        F.avg("price").alias("avg_price"),
        F.min("days_ago").alias("days_since_last_purchase")
    )
)
customer_feats.show(5)

+--------------------+-----------+--------------+--------------------+------------------------+
|         customer_id|n_purchases|n_unique_items|           avg_price|days_since_last_purchase|
+--------------------+-----------+--------------+--------------------+------------------------+
|25add38a9584701b7...|        183|           170|0.029469852736871353|                      36|
|e38cedeb861af12e9...|         11|             5|0.023058551617873648|                     545|
|22e699f8a8d74c1ea...|          6|             6| 0.01834463276836158|                     722|
|c6e8b198cc0e4ddd6...|        299|           219| 0.04250104869338472|                       5|
|b473ffd0c6b98aadd...|         10|             6| 0.03049152542372881|                     707|
+--------------------+-----------+--------------+--------------------+------------------------+
only showing top 5 rows


In [34]:
item_popularity = (
    train.filter(F.col("days_ago") <= 28)
    .groupBy("article_id")
    .agg(F.count("*").alias("purchases_last_28d"))
)
top_items = item_popularity.orderBy(F.desc("purchases_last_28d")).limit(12)
top_items.show()

+----------+------------------+
|article_id|purchases_last_28d|
+----------+------------------+
| 751471001|              2758|
| 706016001|              2408|
| 915526001|              2383|
| 751471043|              2328|
| 918292001|              2327|
| 915529003|              2177|
| 896152002|              2028|
| 898694001|              1836|
| 916468003|              1822|
| 448509014|              1807|
| 863595006|              1767|
| 915526002|              1600|
+----------+------------------+



In [35]:
customer_feats.write.mode("overwrite").parquet("customer_feats.parquet")
item_popularity.write.mode("overwrite").parquet("item_popularity.parquet")

In [36]:
from pyspark.sql.window import Window

In [37]:
user_vocab = (
    train.select("customer_id").distinct()
    .withColumn("user_idx", F.row_number().over(Window.orderBy("customer_id")))
)
print("Users:", user_vocab.count())
user_vocab.show(5, truncate=False)

Users: 1356709
+----------------------------------------------------------------+--------+
|customer_id                                                     |user_idx|
+----------------------------------------------------------------+--------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|1       |
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|2       |
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|3       |
|00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e|4       |
|00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a|5       |
+----------------------------------------------------------------+--------+
only showing top 5 rows


In [38]:
item_vocab = (
    train.select("article_id").distinct()
    .withColumn("item_idx", F.row_number().over(Window.orderBy("article_id")))
)
print("Items:", item_vocab.count())

Items: 103880


In [39]:
item_cat_cols = ["product_type_name", "colour_group_name", "department_name", "garment_group_name"]
for c in item_cat_cols:
    print(c, articles.select(c).distinct().count())

for c in ["club_member_status", "fashion_news_frequency"]:
    print(c, customers_clean.select(c).distinct().count())

product_type_name 131
colour_group_name 50
department_name 250
garment_group_name 21
club_member_status 4
fashion_news_frequency 3


In [40]:
user_table = (
    user_vocab
    .join(customers_clean.select("customer_id", "age", "club_member_status",
                                 "fashion_news_frequency", "age_missing"), on="customer_id", how="left")
    .join(customer_feats, on="customer_id", how="left")
)

item_table = item_vocab.join(
    articles.select("article_id", *item_cat_cols), on="article_id", how="left"
)

In [42]:
user_table.write.mode("overwrite").parquet("user_table.parquet")
item_table.write.mode("overwrite").parquet("item_table.parquet")

In [43]:
!apt-get update -qq
!apt-get install -y -qq redis-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libjemalloc2_5.3.0-2build1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.3.0-2build1) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../liblzf1_3.6-4_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-4) ...
Selecting previously unselected package redis-tools.
Preparing to unpack .../redis-tools_5%3a7.0.15-1ubuntu0.24.04.5_amd64.deb ...
Unpacking redis-tools (5:7.0.15-1ubuntu0.24.04.5) ...
Selecting previously unselected package redis-server.
Preparing to unpack .../redis-server_5%3a7.0.15-1ubuntu0.24.04.5_amd64.deb ...
Unpacking redis-server (5:7.0.15-1ubuntu0.24.04.5) ...
Setting up libjemalloc2:amd64 (5.3.0-2build1) ..

In [44]:
!redis-server --daemonize yes

In [45]:
!redis-cli ping

PONG


In [46]:
!redis-cli SET greeting "hello"
!redis-cli GET greeting

OK
"hello"


In [47]:
!redis-cli HSET user:1 age 32 club_status ACTIVE n_purchases 183
!redis-cli HGETALL user:1
!redis-cli HGET user:1 age

(integer) 3
1) "age"
2) "32"
3) "club_status"
4) "ACTIVE"
5) "n_purchases"
6) "183"
"32"


In [48]:
!pip install -q redis

In [49]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print(r.ping())

True


In [50]:
r.hset("user:2", mapping={"age": 25, "club_status": "ACTIVE", "n_purchases": 11})
print(r.hgetall("user:2"))
print(r.hget("user:2", "age"))

{'age': '25', 'club_status': 'ACTIVE', 'n_purchases': '11'}
25


In [51]:
r.set("fallback:top12", "751471001,706016001,915526001")
print(r.get("fallback:top12"))

751471001,706016001,915526001


In [52]:
import time

start = time.perf_counter()
for _ in range(1000):
    r.hgetall("user:2")
elapsed = time.perf_counter() - start
print("Average per lookup (ms):", elapsed / 1000 * 1000)

Average per lookup (ms): 0.15637531599986687


In [53]:
r.delete("greeting", "user:1", "user:2", "fallback:top12")
print(r.keys("*"))

[]
